# Chapter 1.2: GPU Memory Hierarchy

## Learning Objectives

By the end of this notebook, you will:

1. **Understand** the GPU architecture: SMs, warps, and thread hierarchy
2. **Map** the memory hierarchy: registers, shared memory (SRAM), L2 cache, HBM
3. **Distinguish** between memory bandwidth and compute throughput (FLOPS)
4. **Know** the key bandwidth numbers for A100, H100, and other GPUs
5. **Measure** actual memory bandwidth using PyTorch
6. **Explain** memory layout concepts: row-major, column-major, coalesced access
7. **Identify** where LLM weights and KV-cache reside in the GPU memory hierarchy

---

## Why Does This Matter for LLM Serving?

LLM inference, particularly the decode phase, is **memory-bandwidth-bound**. Understanding the GPU memory hierarchy is crucial because:

- The decode phase reads enormous amounts of data (model weights + KV-cache) but does relatively little computation
- Techniques like FlashAttention exploit the memory hierarchy (using SRAM instead of HBM)
- Quantization reduces memory bandwidth requirements by shrinking data sizes
- Efficient serving requires maximizing bandwidth utilization

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part1/chapter_1.2_gpu_memory.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part1/chapter_1.2_gpu_memory.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import time
from typing import Dict, List

# Check if CUDA is available
if torch.cuda.is_available():
    device = torch.device('cuda')
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / (1024**3)
    print(f"GPU: {gpu_name}")
    print(f"GPU Memory: {gpu_mem:.1f} GB")
else:
    device = torch.device('cpu')
    print("No GPU available. Running on CPU.")
    print("Note: Bandwidth measurements will reflect CPU/RAM, not GPU/HBM.")

---

## 1. GPU Architecture Overview

### 1.1 CPU vs GPU: Fundamental Differences

| Feature | CPU | GPU |
|---------|-----|-----|
| Design Philosophy | Latency-optimized | Throughput-optimized |
| Cores | Few powerful cores (8-64) | Thousands of simple cores (10K+) |
| Clock Speed | High (3-5 GHz) | Lower (1-2 GHz) |
| Cache | Large per-core cache | Small shared cache |
| Control Flow | Excellent branch prediction | Poor branching performance |
| Best For | Sequential, complex tasks | Massively parallel, simple tasks |

### 1.2 GPU Execution Hierarchy

```
GPU Device
 ├── Streaming Multiprocessor (SM) × N   (e.g., 108 SMs in A100)
 │    ├── Warp Scheduler(s)               (schedules groups of 32 threads)
 │    ├── CUDA Cores                      (FP32/INT32 execution units)
 │    ├── Tensor Cores                    (matrix multiply units)
 │    ├── Register File                   (per-thread, fastest)
 │    ├── Shared Memory / L1 Cache        (per-SM, fast SRAM)
 │    └── Load/Store Units
 ├── L2 Cache                             (shared across all SMs)
 └── HBM (High Bandwidth Memory)          (main GPU memory, slowest)
```

#### Key Concepts:

- **Thread**: The smallest execution unit. Each thread has its own registers.
- **Warp**: A group of 32 threads that execute the same instruction simultaneously (SIMT).
- **Thread Block**: A group of threads (up to 1024) that can cooperate via shared memory.
- **Grid**: A collection of thread blocks that execute a kernel.
- **Streaming Multiprocessor (SM)**: The main computing unit. Multiple warps run on one SM.

In [ ]:
# Visualize the GPU execution hierarchy

fig, ax = plt.subplots(1, 1, figsize=(14, 8))
ax.set_xlim(0, 14)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('GPU Execution Hierarchy', fontsize=16, fontweight='bold', pad=20)

# GPU Device
gpu_rect = mpatches.FancyBboxPatch((0.5, 0.5), 13, 9, boxstyle="round,pad=0.1",
                                    facecolor='#ecf0f1', edgecolor='#2c3e50', linewidth=3)
ax.add_patch(gpu_rect)
ax.text(7, 9.2, 'GPU Device', fontsize=14, ha='center', fontweight='bold')

# SMs
colors_sm = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12']
for i in range(4):
    x = 1.2 + i * 3.1
    sm_rect = mpatches.FancyBboxPatch((x, 2.5), 2.8, 6, boxstyle="round,pad=0.05",
                                      facecolor=colors_sm[i], edgecolor='#2c3e50', 
                                      linewidth=1.5, alpha=0.3)
    ax.add_patch(sm_rect)
    ax.text(x + 1.4, 8.1, f'SM {i}', fontsize=11, ha='center', fontweight='bold')
    
    # Warps inside SM
    for j in range(3):
        warp_y = 6.8 - j * 1.5
        warp_rect = mpatches.FancyBboxPatch((x + 0.15, warp_y), 2.5, 1.2,
                                            boxstyle="round,pad=0.03",
                                            facecolor='white', edgecolor='#34495e',
                                            linewidth=1, alpha=0.8)
        ax.add_patch(warp_rect)
        ax.text(x + 1.4, warp_y + 0.6, f'Warp {j} (32 threads)', 
                fontsize=7, ha='center', va='center')
    
    # Shared memory
    smem_rect = mpatches.FancyBboxPatch((x + 0.15, 2.7), 2.5, 0.8,
                                        boxstyle="round,pad=0.03",
                                        facecolor='#f1c40f', edgecolor='#34495e',
                                        linewidth=1, alpha=0.7)
    ax.add_patch(smem_rect)
    ax.text(x + 1.4, 3.1, 'Shared Mem / L1', fontsize=7, ha='center', va='center')

# L2 Cache
l2_rect = mpatches.FancyBboxPatch((1.0, 1.5), 12, 0.7, boxstyle="round,pad=0.03",
                                   facecolor='#9b59b6', edgecolor='#2c3e50',
                                   linewidth=1.5, alpha=0.5)
ax.add_patch(l2_rect)
ax.text(7, 1.85, 'L2 Cache (shared across all SMs)', fontsize=10, ha='center', 
        va='center', fontweight='bold')

# HBM
hbm_rect = mpatches.FancyBboxPatch((1.0, 0.6), 12, 0.7, boxstyle="round,pad=0.03",
                                    facecolor='#e74c3c', edgecolor='#2c3e50',
                                    linewidth=1.5, alpha=0.5)
ax.add_patch(hbm_rect)
ax.text(7, 0.95, 'HBM (High Bandwidth Memory) - Main GPU Memory', fontsize=10, 
        ha='center', va='center', fontweight='bold', color='white')

# "..." for more SMs
ax.text(6.8, 5.0, '... more SMs ...', fontsize=10, ha='center', style='italic', alpha=0.5)

plt.tight_layout()
plt.show()

---

## 2. The Memory Hierarchy

### 2.1 Hierarchy Details

The GPU memory hierarchy follows the classic trade-off: **faster memory is smaller and more expensive**.

| Memory Level | Size (A100) | Bandwidth | Latency | Scope |
|:------------|:-----------|:----------|:--------|:------|
| **Registers** | 256 KB per SM | ~19 TB/s | 0 cycles | Per thread |
| **Shared Memory (SRAM)** | 164 KB per SM (configurable) | ~19 TB/s | ~20 cycles | Per thread block |
| **L2 Cache** | 40 MB | ~5 TB/s | ~200 cycles | All SMs |
| **HBM2e** | 80 GB | 2.0 TB/s | ~400 cycles | All SMs |

**Key insight**: There is a ~10x bandwidth gap between SRAM and HBM. This gap is what FlashAttention exploits!

### 2.2 The SRAM-HBM Gap

```
Registers:    ~19 TB/s   ████████████████████████████████████
Shared/L1:    ~19 TB/s   ████████████████████████████████████
L2 Cache:     ~5  TB/s   █████████████
HBM:          ~2  TB/s   █████
```

For LLM inference:
- Model weights live in **HBM** (too large for SRAM)
- KV-Cache lives in **HBM** (too large and dynamic)
- Intermediate computations ideally happen in **SRAM** (registers + shared memory)

In [ ]:
## Figure: GPU Memory Hierarchy Pyramid
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

fig, ax = plt.subplots(figsize=(12, 8))
ax.set_xlim(0, 14)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('GPU Memory Hierarchy: Speed vs Size Trade-off', fontsize=16, fontweight='bold')

# Pyramid layers (bottom to top: larger & slower to smaller & faster)
layers = [
    {'y': 1.0, 'width': 12, 'height': 1.5, 'label': 'HBM (High Bandwidth Memory)',
     'detail': '80 GB | 2-4.8 TB/s | ~400 cycles', 'color': '#E74C3C'},
    {'y': 3.0, 'width': 9, 'height': 1.5, 'label': 'L2 Cache',
     'detail': '40-50 MB | ~5 TB/s | ~200 cycles', 'color': '#F39C12'},
    {'y': 5.0, 'width': 6, 'height': 1.5, 'label': 'Shared Memory / L1 (SRAM)',
     'detail': '164-228 KB/SM | ~19 TB/s | ~20 cycles', 'color': '#27AE60'},
    {'y': 7.0, 'width': 3.5, 'height': 1.5, 'label': 'Registers',
     'detail': '256 KB/SM | ~19 TB/s | 0 cycles', 'color': '#4A90D9'},
]

for layer in layers:
    cx = 7  # center x
    w = layer['width']
    ax.add_patch(patches.FancyBboxPatch(
        (cx - w/2, layer['y']), w, layer['height'],
        boxstyle="round,pad=0.1", facecolor=layer['color'], alpha=0.8,
        edgecolor='black', linewidth=1.5))
    ax.text(cx, layer['y'] + 0.9, layer['label'],
            ha='center', va='center', fontsize=11, fontweight='bold', color='white')
    ax.text(cx, layer['y'] + 0.35, layer['detail'],
            ha='center', va='center', fontsize=8, color='white', alpha=0.9)

# Side labels
ax.annotate('', xy=(0.8, 8.5), xytext=(0.8, 1.0),
            arrowprops=dict(arrowstyle='<->', lw=2, color='#2C3E50'))
ax.text(0.3, 5, 'FASTER\n(lower\nlatency)', fontsize=9, fontweight='bold', rotation=90,
        ha='center', va='center', color='#4A90D9')

ax.annotate('', xy=(13.2, 1.0), xytext=(13.2, 8.5),
            arrowprops=dict(arrowstyle='<->', lw=2, color='#2C3E50'))
ax.text(13.7, 5, 'LARGER\n(more\ncapacity)', fontsize=9, fontweight='bold', rotation=90,
        ha='center', va='center', color='#E74C3C')

# Where LLM data lives
ax.text(7, 0.3, 'Model weights + KV-Cache live here (too large for SRAM)',
        ha='center', fontsize=9, style='italic', color='#E74C3C', fontweight='bold')
ax.text(7, 9.0, 'FlashAttention keeps attention matrix here (avoids HBM)',
        ha='center', fontsize=9, style='italic', color='#4A90D9', fontweight='bold')

# Bandwidth gap annotation
ax.annotate('~10x bandwidth gap!', xy=(12, 2.5), xytext=(12, 5.5),
            fontsize=10, fontweight='bold', color='#8E44AD', ha='center',
            arrowprops=dict(arrowstyle='->', color='#8E44AD', lw=2))

plt.tight_layout()
plt.show()
print("The 10x SRAM-to-HBM bandwidth gap is what FlashAttention exploits.")
print("LLM inference is memory-bound because weights must be read from HBM for every token.")

In [ ]:
# Visualize the memory hierarchy for different GPUs

gpu_specs = {
    'A100 (80GB)': {
        'hbm_size_gb': 80,
        'hbm_bw_tbs': 2.0,
        'l2_size_mb': 40,
        'sram_per_sm_kb': 164,
        'n_sms': 108,
        'sram_bw_tbs': 19.0,
        'fp16_tflops': 312,
        'fp32_tflops': 19.5,
        'tensor_tflops': 312,  # FP16 Tensor Core
    },
    'H100 (80GB)': {
        'hbm_size_gb': 80,
        'hbm_bw_tbs': 3.35,
        'l2_size_mb': 50,
        'sram_per_sm_kb': 228,
        'n_sms': 132,
        'sram_bw_tbs': 33.0,
        'fp16_tflops': 989,
        'fp32_tflops': 67,
        'tensor_tflops': 989,  # FP16 Tensor Core
    },
    'H200 (141GB)': {
        'hbm_size_gb': 141,
        'hbm_bw_tbs': 4.8,
        'l2_size_mb': 50,
        'sram_per_sm_kb': 228,
        'n_sms': 132,
        'sram_bw_tbs': 33.0,
        'fp16_tflops': 989,
        'fp32_tflops': 67,
        'tensor_tflops': 989,
    },
    'RTX 4090': {
        'hbm_size_gb': 24,
        'hbm_bw_tbs': 1.008,
        'l2_size_mb': 72,
        'sram_per_sm_kb': 128,
        'n_sms': 128,
        'sram_bw_tbs': 15.0,
        'fp16_tflops': 330,
        'fp32_tflops': 82.6,
        'tensor_tflops': 330,
    }
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

gpu_names = list(gpu_specs.keys())
colors = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12']

# HBM Capacity
hbm_sizes = [gpu_specs[g]['hbm_size_gb'] for g in gpu_names]
bars = axes[0].bar(gpu_names, hbm_sizes, color=colors, alpha=0.8)
axes[0].set_ylabel('HBM Size (GB)', fontsize=12)
axes[0].set_title('GPU Memory Capacity', fontsize=14)
for bar, val in zip(bars, hbm_sizes):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
                 f'{val} GB', ha='center', fontweight='bold')
axes[0].tick_params(axis='x', rotation=15)

# HBM Bandwidth
hbm_bws = [gpu_specs[g]['hbm_bw_tbs'] for g in gpu_names]
bars = axes[1].bar(gpu_names, hbm_bws, color=colors, alpha=0.8)
axes[1].set_ylabel('HBM Bandwidth (TB/s)', fontsize=12)
axes[1].set_title('Memory Bandwidth', fontsize=14)
for bar, val in zip(bars, hbm_bws):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.05,
                 f'{val} TB/s', ha='center', fontweight='bold')
axes[1].tick_params(axis='x', rotation=15)

# Compute (Tensor Core TFLOPS)
tflops = [gpu_specs[g]['tensor_tflops'] for g in gpu_names]
bars = axes[2].bar(gpu_names, tflops, color=colors, alpha=0.8)
axes[2].set_ylabel('FP16 Tensor TFLOPS', fontsize=12)
axes[2].set_title('Compute Throughput', fontsize=14)
for bar, val in zip(bars, tflops):
    axes[2].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 10,
                 f'{val}', ha='center', fontweight='bold')
axes[2].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

### 2.3 Memory Bandwidth vs Compute: The Key Ratio

An extremely important metric is the **compute-to-memory ratio** (also called the **ops:byte ratio** or **machine balance point**):

$$\text{Compute-to-Memory Ratio} = \frac{\text{Peak FLOPS}}{\text{Peak Memory Bandwidth}} \quad [\text{FLOPs/byte}]$$

This tells us: for every byte we read from memory, how many operations can the GPU perform?

If our algorithm does fewer operations per byte than this ratio, it's **memory-bound**.
If it does more, it's **compute-bound**.

In [ ]:
print("Compute-to-Memory Ratio (FP16 Tensor Core FLOPS / HBM Bandwidth)")
print("=" * 70)

for gpu_name, specs in gpu_specs.items():
    # Convert TFLOPS to FLOPS, TB/s to bytes/s
    flops = specs['tensor_tflops'] * 1e12
    bw = specs['hbm_bw_tbs'] * 1e12
    ratio = flops / bw
    
    print(f"  {gpu_name:<20} {specs['tensor_tflops']:>6} TFLOPS / {specs['hbm_bw_tbs']:.2f} TB/s "
          f"= {ratio:>6.0f} FLOPs/byte")

print("\n" + "=" * 70)
print("\nInterpretation:")
print("  - If your kernel does < ratio FLOPs per byte read, it's MEMORY-BOUND")
print("  - If your kernel does > ratio FLOPs per byte read, it's COMPUTE-BOUND")
print("\nLLM decode (batch=1): ~1-2 FLOPs/byte  -> heavily MEMORY-BOUND")
print("LLM prefill:          ~100+ FLOPs/byte  -> COMPUTE-BOUND")
print("\nThe gap between hardware ratio (~150-300) and decode ratio (~1-2)")
print("means the GPU compute units are >99% idle during decode!")

---

## 3. Measuring Actual Memory Bandwidth

Let's measure the actual achieved memory bandwidth on your GPU/CPU using simple copy and arithmetic operations.

In [ ]:
def measure_bandwidth(
    device: torch.device,
    sizes_mb: list = [1, 4, 16, 64, 256, 512, 1024],
    n_iterations: int = 100,
    warmup: int = 10
) -> dict:
    """
    Measure memory bandwidth using a simple element-wise copy.
    
    Bandwidth = bytes_transferred / time
    For copy: bytes_transferred = 2 * size (one read + one write)
    """
    results = {}
    
    for size_mb in sizes_mb:
        n_elements = int(size_mb * 1024 * 1024 / 4)  # FP32 = 4 bytes
        
        try:
            a = torch.randn(n_elements, device=device)
            b = torch.empty_like(a)
        except RuntimeError:
            print(f"  Skipping {size_mb} MB (out of memory)")
            continue
        
        # Warmup
        for _ in range(warmup):
            b.copy_(a)
        if device.type == 'cuda':
            torch.cuda.synchronize()
        
        # Measure
        start = time.perf_counter()
        for _ in range(n_iterations):
            b.copy_(a)
        if device.type == 'cuda':
            torch.cuda.synchronize()
        elapsed = time.perf_counter() - start
        
        bytes_transferred = 2 * n_elements * 4 * n_iterations  # read + write, FP32
        bandwidth_gbs = bytes_transferred / elapsed / 1e9
        
        results[size_mb] = bandwidth_gbs
        
        del a, b
        if device.type == 'cuda':
            torch.cuda.empty_cache()
    
    return results


print(f"Measuring bandwidth on {device}...")
bw_results = measure_bandwidth(device, sizes_mb=[1, 4, 16, 64, 256, 512])

print(f"\nBandwidth Results ({device}):")
print(f"{'Size (MB)':<15} {'Bandwidth (GB/s)':<20}")
print("-" * 35)
for size, bw in bw_results.items():
    print(f"{size:<15} {bw:<20.1f}")

In [ ]:
# Plot bandwidth results

fig, ax = plt.subplots(figsize=(10, 6))

sizes = list(bw_results.keys())
bws = list(bw_results.values())

ax.plot(sizes, bws, 'b-o', linewidth=2, markersize=8, label=f'Measured ({device})')

# Add theoretical peak for reference
if device.type == 'cuda':
    # Try to detect GPU type for peak reference
    gpu_name_lower = torch.cuda.get_device_name(0).lower()
    if 'a100' in gpu_name_lower:
        peak_bw = 2000  # A100 80GB: 2 TB/s
    elif 'h100' in gpu_name_lower:
        peak_bw = 3350  # H100: 3.35 TB/s
    elif '4090' in gpu_name_lower:
        peak_bw = 1008  # RTX 4090: 1.008 TB/s
    else:
        peak_bw = None
    
    if peak_bw:
        ax.axhline(y=peak_bw, color='red', linestyle='--', alpha=0.5, 
                   label=f'Theoretical Peak ({peak_bw} GB/s)')

ax.set_xlabel('Data Size (MB)', fontsize=12)
ax.set_ylabel('Bandwidth (GB/s)', fontsize=12)
ax.set_title('Measured Memory Bandwidth vs Data Size', fontsize=14)
ax.set_xscale('log', base=2)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nSmall data sizes may show higher bandwidth (data fits in L2 cache).")
print("Larger data sizes show the actual HBM bandwidth.")

### 3.2 Measuring Bandwidth with Different Operations

Different operations have different memory access patterns. Let's compare.

In [ ]:
def measure_operation_bandwidth(
    device: torch.device,
    size_mb: int = 256,
    n_iterations: int = 100,
    warmup: int = 10
) -> dict:
    """
    Measure effective bandwidth for different operations.
    """
    n_elements = int(size_mb * 1024 * 1024 / 4)  # FP32
    
    results = {}
    
    # 1. Vector copy (read + write)
    a = torch.randn(n_elements, device=device)
    b = torch.empty_like(a)
    
    for _ in range(warmup):
        b.copy_(a)
    if device.type == 'cuda': torch.cuda.synchronize()
    
    start = time.perf_counter()
    for _ in range(n_iterations):
        b.copy_(a)
    if device.type == 'cuda': torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    results['Copy'] = 2 * n_elements * 4 * n_iterations / elapsed / 1e9
    
    # 2. Vector add (2 reads + 1 write)
    c = torch.empty_like(a)
    for _ in range(warmup):
        torch.add(a, b, out=c)
    if device.type == 'cuda': torch.cuda.synchronize()
    
    start = time.perf_counter()
    for _ in range(n_iterations):
        torch.add(a, b, out=c)
    if device.type == 'cuda': torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    results['Add (a+b)'] = 3 * n_elements * 4 * n_iterations / elapsed / 1e9
    
    # 3. Scale (1 read + 1 write)
    for _ in range(warmup):
        torch.mul(a, 2.0, out=b)
    if device.type == 'cuda': torch.cuda.synchronize()
    
    start = time.perf_counter()
    for _ in range(n_iterations):
        torch.mul(a, 2.0, out=b)
    if device.type == 'cuda': torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    results['Scale (a*2)'] = 2 * n_elements * 4 * n_iterations / elapsed / 1e9
    
    # 4. Reduction (sum) - read only
    for _ in range(warmup):
        _ = a.sum()
    if device.type == 'cuda': torch.cuda.synchronize()
    
    start = time.perf_counter()
    for _ in range(n_iterations):
        _ = a.sum()
    if device.type == 'cuda': torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    results['Sum (reduction)'] = n_elements * 4 * n_iterations / elapsed / 1e9
    
    del a, b, c
    if device.type == 'cuda': torch.cuda.empty_cache()
    
    return results


print(f"Measuring operation bandwidths on {device} (256 MB data)...")
op_bw = measure_operation_bandwidth(device, size_mb=256)

print(f"\n{'Operation':<20} {'Effective Bandwidth (GB/s)':<30}")
print("-" * 50)
for op, bw in op_bw.items():
    print(f"{op:<20} {bw:<30.1f}")

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
ops = list(op_bw.keys())
bws = list(op_bw.values())
ax.barh(ops, bws, color=['#3498db', '#2ecc71', '#e74c3c', '#f39c12'], alpha=0.8)
ax.set_xlabel('Effective Bandwidth (GB/s)', fontsize=12)
ax.set_title(f'Memory Bandwidth by Operation Type ({device})', fontsize=14)
for i, (op, bw) in enumerate(zip(ops, bws)):
    ax.text(bw + max(bws)*0.02, i, f'{bw:.0f} GB/s', va='center', fontweight='bold')
plt.tight_layout()
plt.show()

---

## 4. Memory Layout: Row-Major vs Column-Major

### 4.1 How Data is Stored in Memory

Memory is fundamentally a **1D array of bytes**. When we store a 2D matrix, we have two choices:

- **Row-major (C-style)**: Elements in the same row are contiguous in memory
  - `A[0,0], A[0,1], A[0,2], A[1,0], A[1,1], A[1,2], ...`
  - Used by C, C++, PyTorch, NumPy (default)

- **Column-major (Fortran-style)**: Elements in the same column are contiguous
  - `A[0,0], A[1,0], A[2,0], A[0,1], A[1,1], A[2,1], ...`
  - Used by Fortran, MATLAB, Julia

### 4.2 Coalesced Memory Access

GPUs read memory in large chunks called **cache lines** (typically 128 bytes). When 32 threads in a warp each read consecutive memory addresses, this is called **coalesced access** and is extremely efficient.

If threads read scattered addresses, each thread triggers a separate memory transaction, wasting bandwidth.

In [ ]:
# Visualize row-major vs column-major memory layout

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Matrix
matrix = np.array([[1, 2, 3, 4],
                   [5, 6, 7, 8],
                   [9, 10, 11, 12]])

# Show the matrix
ax = axes[0]
for i in range(3):
    for j in range(4):
        color = plt.cm.Set3(i / 3)
        rect = plt.Rectangle((j, 2-i), 1, 1, facecolor=color, edgecolor='black', linewidth=1.5)
        ax.add_patch(rect)
        ax.text(j + 0.5, 2.5 - i, str(matrix[i, j]), ha='center', va='center', fontsize=14, fontweight='bold')
ax.set_xlim(0, 4)
ax.set_ylim(0, 3)
ax.set_title('Matrix A (3x4)', fontsize=14)
ax.set_xlabel('Columns')
ax.set_ylabel('Rows')
ax.set_aspect('equal')

# Row-major layout
ax = axes[1]
row_major = matrix.flatten()  # default in NumPy/PyTorch
for i, val in enumerate(row_major):
    row_idx = i // 4
    color = plt.cm.Set3(row_idx / 3)
    rect = plt.Rectangle((i, 0), 1, 1, facecolor=color, edgecolor='black', linewidth=1.5)
    ax.add_patch(rect)
    ax.text(i + 0.5, 0.5, str(val), ha='center', va='center', fontsize=12, fontweight='bold')
    ax.text(i + 0.5, -0.3, f'[{i}]', ha='center', va='center', fontsize=8, color='gray')
ax.set_xlim(0, 12)
ax.set_ylim(-0.5, 1.5)
ax.set_title('Row-Major (C/PyTorch)', fontsize=14)
ax.set_xlabel('Memory Address')
ax.set_yticks([])

# Column-major layout
ax = axes[2]
col_major = matrix.flatten(order='F')
for i, val in enumerate(col_major):
    col_idx = i // 3
    color = plt.cm.Pastel1(col_idx / 4)
    rect = plt.Rectangle((i, 0), 1, 1, facecolor=color, edgecolor='black', linewidth=1.5)
    ax.add_patch(rect)
    ax.text(i + 0.5, 0.5, str(val), ha='center', va='center', fontsize=12, fontweight='bold')
    ax.text(i + 0.5, -0.3, f'[{i}]', ha='center', va='center', fontsize=8, color='gray')
ax.set_xlim(0, 12)
ax.set_ylim(-0.5, 1.5)
ax.set_title('Column-Major (Fortran)', fontsize=14)
ax.set_xlabel('Memory Address')
ax.set_yticks([])

plt.tight_layout()
plt.show()

print("Row-major:    Row 0 elements are contiguous → iterating over columns in a row is fast")
print("Column-major: Column 0 elements are contiguous → iterating over rows in a column is fast")

In [ ]:
# Demonstrate the performance impact of memory access patterns

def measure_access_pattern(device, N=4096, n_iterations=50, warmup=5):
    """
    Compare row-wise vs column-wise access performance.
    """
    matrix = torch.randn(N, N, device=device)
    
    # Row-wise sum (contiguous access in row-major)
    for _ in range(warmup):
        _ = matrix.sum(dim=1)  # sum across columns (within each row)
    if device.type == 'cuda': torch.cuda.synchronize()
    
    start = time.perf_counter()
    for _ in range(n_iterations):
        _ = matrix.sum(dim=1)
    if device.type == 'cuda': torch.cuda.synchronize()
    row_time = (time.perf_counter() - start) / n_iterations
    
    # Column-wise sum (strided access in row-major)
    for _ in range(warmup):
        _ = matrix.sum(dim=0)  # sum across rows (within each column)
    if device.type == 'cuda': torch.cuda.synchronize()
    
    start = time.perf_counter()
    for _ in range(n_iterations):
        _ = matrix.sum(dim=0)
    if device.type == 'cuda': torch.cuda.synchronize()
    col_time = (time.perf_counter() - start) / n_iterations
    
    # Transposed column-wise sum (contiguous after transpose)
    matrix_t = matrix.t().contiguous()
    for _ in range(warmup):
        _ = matrix_t.sum(dim=1)
    if device.type == 'cuda': torch.cuda.synchronize()
    
    start = time.perf_counter()
    for _ in range(n_iterations):
        _ = matrix_t.sum(dim=1)
    if device.type == 'cuda': torch.cuda.synchronize()
    transposed_time = (time.perf_counter() - start) / n_iterations
    
    return {
        'Row-wise sum (contiguous)': row_time * 1000,
        'Column-wise sum (strided)': col_time * 1000,
        'Transposed column sum': transposed_time * 1000
    }


results = measure_access_pattern(device)

print(f"\nMemory Access Pattern Performance ({device}, 4096x4096 matrix):")
print("=" * 50)
for name, time_ms in results.items():
    print(f"  {name:<35} {time_ms:.3f} ms")

print(f"\nColumn-wise is {results['Column-wise sum (strided)'] / results['Row-wise sum (contiguous)']:.2f}x "
      f"slower than row-wise")
print("This shows the importance of contiguous memory access!")

---

## 5. Where LLM Data Lives in GPU Memory

### 5.1 Memory Allocation Breakdown

During LLM inference, the GPU memory is used for:

| Component | Location | Size | Lifetime |
|-----------|----------|------|----------|
| Model weights | HBM | Fixed (e.g., 14 GB for 7B FP16) | Persistent |
| KV-Cache | HBM | Grows with seq_len × batch_size | Per request |
| Activations | HBM + SRAM | ~1-5% of total | Transient (per layer) |
| CUDA context | HBM | ~0.5-2 GB | Persistent |
| Intermediate buffers | SRAM (ideally) | Small | Per operation |

### 5.2 The Memory Flow During Inference

```
Decode step (single token):

HBM                          SRAM (SM)                     Output
┌─────────────┐             ┌──────────┐
│ Model Weights│ ──read──>  │ Compute  │ ──write──>  HBM (output)
│  (14 GB)    │             │ (matmul) │             (tiny: 1 token)
└─────────────┘             └──────────┘
┌─────────────┐                  ^
│  KV-Cache   │ ──read──────────/
│ (variable)  │
└─────────────┘

Bottleneck: Reading model weights from HBM (14 GB every single token!)
```

In [ ]:
# Visualize memory allocation during inference

def plot_memory_allocation(
    model_name: str,
    model_weights_gb: float,
    kv_cache_gb: float,
    activations_gb: float,
    cuda_overhead_gb: float,
    gpu_total_gb: float
):
    fig, ax = plt.subplots(figsize=(12, 3))
    
    components = [
        ('Model Weights', model_weights_gb, '#3498db'),
        ('KV-Cache', kv_cache_gb, '#e74c3c'),
        ('Activations', activations_gb, '#2ecc71'),
        ('CUDA Overhead', cuda_overhead_gb, '#95a5a6'),
    ]
    
    used = sum(c[1] for c in components)
    free = gpu_total_gb - used
    components.append(('Free', max(0, free), '#ecf0f1'))
    
    left = 0
    for name, size, color in components:
        bar = ax.barh(0, size, left=left, color=color, edgecolor='white', linewidth=2, height=0.6)
        if size > gpu_total_gb * 0.05:  # Only label if big enough
            ax.text(left + size/2, 0, f'{name}\n{size:.1f} GB', 
                    ha='center', va='center', fontsize=9, fontweight='bold')
        left += size
    
    ax.set_xlim(0, gpu_total_gb)
    ax.set_xlabel('GPU Memory (GB)', fontsize=12)
    ax.set_title(f'{model_name} - GPU Memory Allocation ({gpu_total_gb} GB GPU)', fontsize=14)
    ax.set_yticks([])
    
    pct_used = used / gpu_total_gb * 100
    ax.text(gpu_total_gb * 0.5, -0.5, f'Used: {used:.1f} GB / {gpu_total_gb} GB ({pct_used:.0f}%)',
            ha='center', fontsize=11)
    
    return fig


# Scenario 1: Llama-2-7B, batch=1, seq=2K
fig = plot_memory_allocation('Llama-2-7B (batch=1, seq=2K)', 14.0, 0.5, 0.5, 1.5, 80)
plt.tight_layout()
plt.show()

# Scenario 2: Llama-2-7B, batch=64, seq=2K
fig = plot_memory_allocation('Llama-2-7B (batch=64, seq=2K)', 14.0, 32.0, 2.0, 1.5, 80)
plt.tight_layout()
plt.show()

# Scenario 3: Llama-2-70B on single GPU (impossible without quantization!)
fig = plot_memory_allocation('Llama-2-70B (batch=1, seq=2K) - OOM!', 140.0, 1.6, 2.0, 1.5, 80)
plt.tight_layout()
plt.show()

print("\nKey observations:")
print("  1. For batch=1, model weights dominate memory usage")
print("  2. For large batches, KV-Cache becomes the dominant consumer")
print("  3. Large models need multiple GPUs (tensor parallelism) or quantization")

---

## 6. Bandwidth-Limited Operations in LLM Inference

### 6.1 Why Decode is Memory-Bound

During decode (generating one token), the model must:

1. **Read all model weights** from HBM (~14 GB for a 7B model in FP16)
2. **Read the KV-Cache** for all previous tokens
3. **Perform computation** (matrix multiply, attention, etc.)
4. **Write the output** (just one token's hidden state)

The computation for a single token is minimal, but reading 14 GB of weights is expensive!

$$\text{Time per token} \geq \frac{\text{Model Weights Size}}{\text{HBM Bandwidth}}$$

For Llama-2-7B on A100:
$$\text{Time} \geq \frac{14 \text{ GB}}{2.0 \text{ TB/s}} = 7 \text{ ms}$$

This gives a theoretical maximum of ~143 tokens/second for batch=1!

In [ ]:
# Calculate theoretical maximum decode throughput

def theoretical_decode_throughput(
    model_size_gb: float,
    hbm_bandwidth_tbs: float,
    batch_size: int = 1
) -> dict:
    """
    Calculate theoretical maximum decode speed (memory-bound limit).
    
    For batch_size > 1, we amortize the weight reading:
    - We still read all weights once
    - But we process batch_size tokens
    - So throughput increases linearly with batch size (until compute-bound)
    """
    bandwidth_gbs = hbm_bandwidth_tbs * 1000  # Convert TB/s to GB/s
    
    # Time to read all weights
    time_read_weights_ms = (model_size_gb / bandwidth_gbs) * 1000
    
    # Tokens per second (batch_size tokens per weight-read)
    tokens_per_sec = batch_size / (time_read_weights_ms / 1000)
    
    # Per-token latency
    per_token_ms = time_read_weights_ms / batch_size
    
    return {
        'weight_read_time_ms': time_read_weights_ms,
        'tokens_per_sec': tokens_per_sec,
        'per_token_latency_ms': per_token_ms,
    }


print("Theoretical Maximum Decode Throughput (Memory-Bound Limit)")
print("=" * 80)

scenarios = [
    ('Llama-2-7B (FP16)', 14.0),
    ('Llama-2-7B (INT8)', 7.0),
    ('Llama-2-7B (INT4)', 3.5),
    ('Llama-2-13B (FP16)', 26.0),
    ('Llama-2-70B (FP16, 1 GPU)', 140.0),
]

gpus = [
    ('A100-80GB', 2.0),
    ('H100-80GB', 3.35),
    ('H200-141GB', 4.8),
]

for model_name, model_gb in scenarios:
    print(f"\n{model_name} ({model_gb:.1f} GB):")
    for gpu_name, bw in gpus:
        result = theoretical_decode_throughput(model_gb, bw, batch_size=1)
        print(f"  {gpu_name}: {result['weight_read_time_ms']:.1f} ms/token "
              f"= {result['tokens_per_sec']:.0f} tok/s (batch=1)")

In [ ]:
# How batching helps: throughput vs batch size

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

batch_sizes = np.arange(1, 129)
model_size_gb = 14.0  # Llama-2-7B FP16

for gpu_name, bw in gpus:
    throughputs = []
    latencies = []
    for bs in batch_sizes:
        result = theoretical_decode_throughput(model_size_gb, bw, int(bs))
        throughputs.append(result['tokens_per_sec'])
        latencies.append(result['per_token_latency_ms'])
    
    axes[0].plot(batch_sizes, throughputs, linewidth=2, label=f'{gpu_name}')
    axes[1].plot(batch_sizes, latencies, linewidth=2, label=f'{gpu_name}')

axes[0].set_xlabel('Batch Size', fontsize=12)
axes[0].set_ylabel('Tokens/second', fontsize=12)
axes[0].set_title('Llama-2-7B: Theoretical Max Decode Throughput', fontsize=14)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)
axes[0].set_yscale('log')

axes[1].set_xlabel('Batch Size', fontsize=12)
axes[1].set_ylabel('Per-Token Latency (ms)', fontsize=12)
axes[1].set_title('Per-Token Latency with Batching', fontsize=14)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nKey insight: Batching amortizes the cost of reading model weights.")
print("With batch=64, you get 64x throughput with only ~1x latency increase.")
print("This is why batching is crucial for serving efficiency!")
print("\n(In practice, throughput plateaus when you become compute-bound or run out of KV-cache memory)")

---

## 7. The FlashAttention Insight: Exploiting the Memory Hierarchy

Traditional attention computation:
1. Compute $S = QK^T$ → write to HBM (size: $n \times n$)
2. Compute $P = \text{softmax}(S)$ → read from HBM, write back to HBM
3. Compute $O = PV$ → read from HBM

Total HBM access: $O(n^2)$ — the attention matrix is materialized in HBM!

**FlashAttention insight**: Keep the attention matrix in **SRAM** by computing it in tiles:
- Process blocks of Q, K, V at a time
- Never write the full $n \times n$ attention matrix to HBM
- Use online softmax to compute correct results without seeing all values

This reduces HBM access from $O(n^2)$ to $O(n)$!

In [ ]:
# Estimate HBM reads/writes for standard vs FlashAttention

def attention_hbm_access(seq_len, d_model, n_heads, method='standard'):
    """
    Estimate total HBM reads/writes for attention computation.
    """
    d_head = d_model // n_heads
    bytes_per_element = 2  # FP16
    
    if method == 'standard':
        # Read Q, K: 2 * seq_len * d_head * bytes
        # Write S = QK^T: seq_len * seq_len * bytes
        # Read S for softmax: seq_len * seq_len * bytes
        # Write P (softmax result): seq_len * seq_len * bytes
        # Read P, V for PV: seq_len * seq_len + seq_len * d_head
        # Write O: seq_len * d_head
        total = (2 * seq_len * d_head +  # read Q, K
                 seq_len * seq_len +      # write S
                 seq_len * seq_len +      # read S
                 seq_len * seq_len +      # write P
                 seq_len * seq_len +      # read P
                 seq_len * d_head +       # read V
                 seq_len * d_head)        # write O
        total *= bytes_per_element * n_heads
        return total
    
    elif method == 'flash':
        # Read Q, K, V: 3 * seq_len * d_head * bytes
        # Write O: seq_len * d_head * bytes
        # No intermediate attention matrix in HBM!
        total = (3 * seq_len * d_head +   # read Q, K, V
                 seq_len * d_head)         # write O
        total *= bytes_per_element * n_heads
        return total


seq_lengths = [512, 1024, 2048, 4096, 8192, 16384]
d_model = 4096  # Llama-2-7B
n_heads = 32

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

standard_bytes = [attention_hbm_access(s, d_model, n_heads, 'standard') / 1e9 for s in seq_lengths]
flash_bytes = [attention_hbm_access(s, d_model, n_heads, 'flash') / 1e9 for s in seq_lengths]

axes[0].plot(seq_lengths, standard_bytes, 'r-o', linewidth=2, label='Standard Attention')
axes[0].plot(seq_lengths, flash_bytes, 'b-o', linewidth=2, label='FlashAttention')
axes[0].set_xlabel('Sequence Length', fontsize=12)
axes[0].set_ylabel('HBM Access (GB)', fontsize=12)
axes[0].set_title('HBM Access: Standard vs Flash Attention', fontsize=14)
axes[0].legend(fontsize=12)
axes[0].grid(True, alpha=0.3)
axes[0].set_yscale('log')

speedups = [s / f for s, f in zip(standard_bytes, flash_bytes)]
axes[1].plot(seq_lengths, speedups, 'g-o', linewidth=2)
axes[1].set_xlabel('Sequence Length', fontsize=12)
axes[1].set_ylabel('HBM Access Reduction (x)', fontsize=12)
axes[1].set_title('FlashAttention HBM Reduction Factor', fontsize=14)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("FlashAttention reduces HBM access dramatically for long sequences.")
print("The savings grow with sequence length because the O(n^2) attention matrix is avoided.")

---

## 8. Practical: Profiling Memory Access Patterns

Let's build a simple profiler to understand how different operations use memory.

In [ ]:
def profile_operation(name, op_fn, input_bytes, output_bytes, flops, 
                      n_iterations=100, warmup=10, device=device):
    """
    Profile an operation and compute its arithmetic intensity.
    """
    # Warmup
    for _ in range(warmup):
        op_fn()
    if device.type == 'cuda': torch.cuda.synchronize()
    
    # Measure
    start = time.perf_counter()
    for _ in range(n_iterations):
        op_fn()
    if device.type == 'cuda': torch.cuda.synchronize()
    elapsed = (time.perf_counter() - start) / n_iterations
    
    total_bytes = input_bytes + output_bytes
    arithmetic_intensity = flops / total_bytes if total_bytes > 0 else 0
    achieved_flops = flops / elapsed
    achieved_bw = total_bytes / elapsed
    
    return {
        'name': name,
        'time_ms': elapsed * 1000,
        'arithmetic_intensity': arithmetic_intensity,
        'achieved_tflops': achieved_flops / 1e12,
        'achieved_bw_gbs': achieved_bw / 1e9,
        'total_bytes': total_bytes,
        'flops': flops
    }


# Profile different operations
N = 4096

# 1. Matrix-Vector multiply (typical decode: weight * hidden_state)
W = torch.randn(N, N, device=device, dtype=torch.float32)
x = torch.randn(N, 1, device=device, dtype=torch.float32)
result_matvec = profile_operation(
    'MatVec (Nx1)',
    lambda: torch.mm(W, x),
    input_bytes=(N*N + N) * 4,   # read W + x
    output_bytes=N * 4,          # write result
    flops=2 * N * N              # 2*N*N FLOPs for matvec
)

# 2. Matrix-Matrix multiply (typical prefill: weight * batch_hidden)
B = torch.randn(N, N, device=device, dtype=torch.float32)
result_matmul = profile_operation(
    'MatMul (NxN)',
    lambda: torch.mm(W, B),
    input_bytes=2 * N * N * 4,   # read W + B
    output_bytes=N * N * 4,      # write result
    flops=2 * N * N * N          # 2*N^3 FLOPs
)

# 3. Element-wise operation (typical: activation, layernorm)
a = torch.randn(N * N, device=device, dtype=torch.float32)
result_elemwise = profile_operation(
    'SiLU activation',
    lambda: F.silu(a),
    input_bytes=N * N * 4,
    output_bytes=N * N * 4,
    flops=N * N * 4  # approximate FLOPs for SiLU
)

# 4. Softmax
s = torch.randn(32, N, N, device=device, dtype=torch.float32)  # attention scores
result_softmax = profile_operation(
    'Softmax',
    lambda: F.softmax(s, dim=-1),
    input_bytes=32 * N * N * 4,
    output_bytes=32 * N * N * 4,
    flops=32 * N * N * 5  # approximate: exp, sum, div per element
)

results = [result_matvec, result_matmul, result_elemwise, result_softmax]

print(f"{'Operation':<20} {'Time (ms)':<12} {'AI (FLOP/B)':<14} {'TFLOPS':<10} {'BW (GB/s)':<12}")
print("-" * 68)
for r in results:
    print(f"{r['name']:<20} {r['time_ms']:<12.3f} {r['arithmetic_intensity']:<14.1f} "
          f"{r['achieved_tflops']:<10.3f} {r['achieved_bw_gbs']:<12.1f}")

print("\nNote: AI = Arithmetic Intensity (FLOP/Byte)")
print("Low AI → memory-bound, High AI → compute-bound")
print("\nMatVec has very low AI (~2) → memory-bound (like LLM decode)")
print("MatMul has high AI (~N/3) → compute-bound (like LLM prefill)")

---

## Exercises

### Exercise 1: Profile Memory Access Patterns

Write a function that measures the effective bandwidth when accessing memory with different stride patterns.

In [ ]:
def measure_strided_bandwidth(device, N=1024*1024, strides=[1, 2, 4, 8, 16, 32], n_iter=50):
    """
    Exercise: Measure bandwidth when reading every k-th element.
    
    For stride=1: contiguous access (coalesced on GPU)
    For stride>1: strided access (uncoalesced, lower bandwidth)
    
    Create a large tensor, then measure the time to sum every k-th element.
    Calculate effective bandwidth as: elements_read * bytes_per_element / time
    """
    # YOUR CODE HERE
    results = {}
    
    # Hint:
    # data = torch.randn(N * max(strides), device=device)
    # for stride in strides:
    #     sliced = data[::stride][:N]  # every stride-th element, take N elements
    #     measure time of sliced.sum()
    #     compute bandwidth
    
    return results

# Test:
# results = measure_strided_bandwidth(device)
# for stride, bw in results.items():
#     print(f"Stride {stride}: {bw:.1f} GB/s")

### Exercise 2: Memory Budget Calculator

Build a comprehensive memory budget calculator that, given a model configuration and GPU specs, determines:
- Maximum batch size
- Maximum sequence length
- Theoretical decode throughput

In [ ]:
def memory_budget_calculator(
    # Model params
    n_params_billion: float,
    n_layers: int,
    n_kv_heads: int,
    d_head: int,
    dtype_bytes: int = 2,  # FP16
    # GPU params
    gpu_memory_gb: float = 80,
    gpu_bandwidth_tbs: float = 2.0,
    overhead_gb: float = 2.0,
    # Serving params
    target_batch_size: int = 32,
    target_seq_len: int = 4096
):
    """
    Exercise: Build a memory budget calculator.
    
    Calculate:
    1. Model weight memory
    2. KV-Cache memory for target batch/seq
    3. Whether it fits in GPU memory
    4. Maximum batch size at target seq_len
    5. Maximum seq_len at target batch_size  
    6. Theoretical decode tokens/s
    """
    # YOUR CODE HERE
    pass

# Test:
# memory_budget_calculator(
#     n_params_billion=7, n_layers=32, n_kv_heads=32, d_head=128,
#     gpu_memory_gb=80, gpu_bandwidth_tbs=2.0
# )

### Exercise 3: Estimate Decode Latency

Given a model and GPU, estimate the minimum decode latency per token. Consider:
- Weight reading time (HBM bandwidth)
- KV-Cache reading time
- Compute time
- Which one is the bottleneck?

In [ ]:
def estimate_decode_latency(
    model_size_gb: float,
    kv_cache_size_gb: float,
    compute_flops: float,  # FLOPs per token
    hbm_bandwidth_tbs: float,
    peak_tflops: float,
    batch_size: int = 1
):
    """
    Exercise: Estimate per-token decode latency.
    
    Calculate:
    - Memory time = (model_size + kv_cache_size) / bandwidth
    - Compute time = compute_flops * batch_size / peak_flops
    - Bottleneck = max(memory_time, compute_time)
    
    Return the breakdown and identify the bottleneck.
    """
    # YOUR CODE HERE
    pass

# Test:
# estimate_decode_latency(
#     model_size_gb=14.0, kv_cache_size_gb=0.5,
#     compute_flops=14e9,  # ~2 * n_params FLOPs per token
#     hbm_bandwidth_tbs=2.0, peak_tflops=312
# )

---

## Summary

### Key Takeaways

1. **GPU Architecture**: GPUs have thousands of simple cores organized into SMs, executing in warps of 32 threads. They are throughput-optimized, not latency-optimized.

2. **Memory Hierarchy**: Registers (fastest) > Shared Memory/L1 (SRAM) > L2 Cache > HBM (slowest but largest). The SRAM-to-HBM bandwidth gap is ~10x.

3. **Bandwidth vs Compute**: Modern GPUs have a compute-to-memory ratio of 150-300 FLOP/byte. LLM decode does only ~1-2 FLOP/byte, making it severely memory-bound.

4. **Memory Layout Matters**: Contiguous (coalesced) memory access is crucial for GPU performance. Row-major storage means iterating over the last dimension is fastest.

5. **LLM Inference Memory**: Model weights and KV-cache live in HBM. Decode reads all weights per token, making HBM bandwidth the bottleneck.

6. **Batching Amortizes Bandwidth**: Increasing batch size amortizes the cost of reading model weights across multiple tokens.

7. **FlashAttention exploits SRAM**: By keeping the attention matrix in SRAM instead of HBM, FlashAttention avoids the $O(n^2)$ HBM bottleneck.

### What's Next

In **Chapter 1.3: Arithmetic Intensity & Roofline Model**, we'll formalize the relationship between memory bandwidth and compute throughput using the **Roofline Model** — a powerful tool for understanding whether an operation is memory-bound or compute-bound, and how close it is to peak performance.